In [1]:
%load_ext autoreload
%autoreload 2
from experiments.replications import (
    run_replications,
    summarize_ci,
    compare_policies,
    labeled_policy_grid,
)
from policy.queue.fifo import FIFOQueuePolicy
from policy.queue.lowest_soc_diff import LowestSoCDiffQueuePolicy
from policy.queue.closest_power_match import ClosestPowerMatchQueuePolicy
from policy.power.proportional import ProportionalPower
from policy.power.static import StaticPower
import pandas as pd


In [ ]:
# --- inputs ---
# mx_wait is filled from the first FIFO run, then applied to other queue rules.
mx_wait = None

queue_policies = [
    FIFOQueuePolicy(),
    LowestSoCDiffQueuePolicy(),
]
#    ClosestPowerMatchQueuePolicy(),
#]
queue_names = ["FIFO", "LSoCD"] #, , "L_SoC_D", "P_Match"]

# Parallel list: one name per power-policy instance (same idea as queue_names).
# One power policy → scenario labels stay queue names (FIFO, L_SoC_D, ...).
# Several power policies and several queue policies → FIFO_Prop, FIFO_Static, ...
power_policies = [
    #StaticPower(),
    ProportionalPower()
    # StaticPower(),  # equal module split; leftovers by unmet p_req
]
power_names = ["Prop"]
#power_names = ["Prop", "Static"]  # with both: FIFO_Prop, FIFO_Static, ...

scenario = {
    "n_piles": 1,
    "n_connectors": 2,
    "n_modules": 6,
    "p_module": 25,
    "queue_capacity": 100,
    "mean_interarrival": 15,
    "delta_arr": None,  # None: continuous; 1: nearest minute
    # None -> config.WARMUP_PERIOD (0.0 out of the box, no warm-up). See
    # sim_benchmark.ipynb for the full usage example. run_replications
    # below always reports BOTH windows regardless (each metric plus its
    # "{name} (measured)" post-warm-up counterpart, from the same episode
    # at no extra cost) -- they're just identical whenever warmup_period=0.
    "warmup_period": None,
    "flush_queue_at_warmup": False,
}

n_reps = 30
confidence = 0.95
seed0 = 0

grid = labeled_policy_grid(
    queue_policies, queue_names, power_policies, power_names
)

# --- run ---
reps = {}
summaries = {}
mean_cols = {}

for label, q_pol, p_pol, q_name, _p_name in grid:
    if q_name != "FIFO" and mx_wait:
        q_pol.max_wait = mx_wait * 0.75
    r = run_replications(
        scenario={**scenario, "queue_policy": q_pol, "power_policy": p_pol},
        n_reps=n_reps,
        seed0=seed0,
    )
    s = summarize_ci(r, confidence=confidence)
    if q_name == "FIFO" and mx_wait is None:
        mx_wait = s.loc["max wait time", "mean"]

    reps[label] = r
    summaries[label] = s
    mean_cols[f"{label}_mean"] = s["mean"]

# paired CRN comparisons (same seeds)
signs, pair_details = compare_policies(reps, confidence=confidence)


In [21]:
pd.DataFrame(mean_cols)

,FIFO_mean,LSoCD_mean
metric,,
finished EVs,34.433333,34.733333
dropped EVs,0.000000,0.000000
total energy delivered (kWh),1657.394856,1658.648343
average L,8.471434,8.227443
average Q,6.566102,6.321762
avg wait time,90.432432,84.807326
max wait time,180.957996,196.223478
avg charge time,38.522479,38.251809
avg sys time,128.954911,123.059135


In [22]:
signs  # -1: first policy lower; 0: n.s.; 1: second policy lower


,FIFO|LSoCD
metric,
finished EVs,-1
dropped EVs,0
total energy delivered (kWh),0
average L,1
average Q,1
avg wait time,1
max wait time,-1
avg charge time,1
avg sys time,1
